# Assignment 3 — AI Product Advertisement Generator

**Objective:** Create a Stable Diffusion image-to-image system that generates multiple professional advertisements for the same headphone product under different environments, lighting, camera angles and visual styles.

This notebook follows the assignment specification:
- Reference product image
- Stable Diffusion image-to-image
- Environment changes
- Camera changes
- Lighting changes
- Style changes
- Advertisement generation
- Comparison of results
- Parameter experiments
- Best prompt + best parameter combination

**Suggested product:** Car  
You can replace it with a smartphone, shoes, watch, laptop, beverage bottle, cosmetic product, etc.

> **Colab setup:** Runtime → Change runtime type → select **GPU**.

> **Reference product used:** the uploaded brown-and-silver premium over-ear headphones.
> All advertisement prompts below are designed to preserve this product identity while changing the environment, camera, lighting and style.


In [ ]:
# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================
!pip -q install -U diffusers transformers accelerate safetensors pillow pandas matplotlib

In [ ]:
# ============================================================
# 2. IMPORT LIBRARIES AND CHECK GPU
# ============================================================
import os
import gc
import torch
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display
from google.colab import files

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Enable a Colab GPU before running the generation cells.")

## 3. Load the Original Reference Product Image

The uploaded reference product is a **premium over-ear headphone**. It is embedded in this notebook so the Colab workflow can use the exact reference image directly.


In [ ]:
# ============================================================
# 3. LOAD EMBEDDED REFERENCE PRODUCT IMAGE
# ============================================================
import base64
from pathlib import Path
from PIL import Image
from IPython.display import display

reference_path = "/content/reference_headphones.jpg"
image_b64 = """..."""

with open(reference_path, "wb") as f:
    f.write(base64.b64decode(image_b64))

reference_image = Image.open(reference_path).convert("RGB")
print("Reference product: Premium over-ear headphones")
print("Reference size:", reference_image.size)
display(reference_image)


In [ ]:
# ============================================================
# 3. UPLOAD REFERENCE PRODUCT IMAGE
# ============================================================
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No image uploaded.")

reference_path = next(iter(uploaded.keys()))
reference_image = Image.open(reference_path).convert("RGB")

# Resize while preserving aspect ratio; 768x512 works well for a landscape advertisement.
reference_image.thumbnail((768, 768))

print("Reference image:", reference_path)
print("Reference size:", reference_image.size)
display(reference_image)

## 4. Load Stable Diffusion Image-to-Image

The pipeline below uses Stable Diffusion through Hugging Face Diffusers.

If the model repository asks for authentication in your Colab session, create a Hugging Face access token and enter it when prompted. Do not publish your token in the submitted notebook.

In [ ]:
# ============================================================
# 4. LOAD STABLE DIFFUSION IMG2IMG
# ============================================================
from diffusers import StableDiffusionImg2ImgPipeline

MODEL_ID = "runwayml/stable-diffusion-v1-5"

# If the model requires authentication in your environment, uncomment:
# from huggingface_hub import login
# login()

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None
)

if torch.cuda.is_available():
    pipe = pipe.to("cuda")
    pipe.enable_attention_slicing()

print("Stable Diffusion image-to-image pipeline loaded.")

## 5. Prompt Design

The same headphone product identity is kept in every generation while environment, camera, lighting and style are changed.

A useful advertisement prompt contains:
1. Product identity
2. Environment
3. Camera description
4. Lighting
5. Commercial/ad quality
6. Photorealistic details

In [ ]:
# ============================================================
# 5. BASE PROMPT AND NEGATIVE PROMPTS
# ============================================================
PRODUCT = "the same premium over-ear headphones shown in the reference image, with brown leather headband and ear cushions, silver metallic ear cups, and circular ear-cup details"

BASE_QUALITY = (
    "professional premium headphone advertisement, premium commercial photography, "
    "highly detailed, photorealistic, realistic materials, realistic reflections, "
    "sharp product details, natural perspective, clean composition, 8k"
)

BASIC_NEGATIVE = (
    "blurry, low quality, distorted, deformed, duplicate headphones, extra ear cups, "
    "extra ear cups, malformed headphones, text, watermark, logo distortion, "
    "cropped product, unrealistic proportions"
)

DETAILED_NEGATIVE = (
    "blurry, low resolution, noisy, oversaturated, headphonestoon, illustration, CGI look, "
    "deformed headphones, malformed product surfaces, duplicate headphones, extra ear cups, extra ear cushions, "
    "extra ear cups, distorted windows, distorted mirrors, warped perspective, "
    "floating object, cut-off product, text, watermark, unwanted letters, "
    "unrealistic reflections, unrealistic shadows"
)

BASE_PROMPT = f"{PRODUCT}, {BASE_QUALITY}"
print(BASE_PROMPT)

## 6. Image Generation Function

`strength` is the image-to-image strength:
- Low strength → preserves the reference product appearance more strongly.
- High strength → allows Stable Diffusion to change the image more aggressively.

The assignment specifically asks for experiments using strengths **0.2, 0.45 and 0.7**.

In [ ]:
# ============================================================
# 6. IMAGE-TO-IMAGE GENERATION FUNCTION
# ============================================================
def generate_ad(
    prompt,
    negative_prompt=BASIC_NEGATIVE,
    seed=42,
    steps=30,
    guidance=7,
    strength=0.45,
    image=reference_image
):
    generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu")
    generator = generator.manual_seed(seed)

    # Use a landscape canvas suitable for advertisements.
    init_image = image.copy()
    init_image = init_image.resize((768, 512))

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        strength=strength,
        guidance_scale=guidance,
        num_inference_steps=steps,
        generator=generator
    ).images[0]

    return result


def show_result(title, image):
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(title)
    plt.show()

# 7. Main Advertisement Variations

The following cells generate the same headphone product in different environments, camera angles, lighting conditions and styles.

In [ ]:
# ============================================================
# 7A. ENVIRONMENT VARIATIONS
# ============================================================
environment_prompts = {
    "Campus": f"{PRODUCT}, parked on a modern university campus with contemporary academic buildings, landscaped greenery, {BASE_QUALITY}",
    "Mountain": f"{PRODUCT}, driving on a scenic mountain road surrounded by dramatic mountains and green landscape, {BASE_QUALITY}",
    "City": f"{PRODUCT}, parked on a sophisticated modern city street with premium architecture and urban background, {BASE_QUALITY}",
    "Beach": f"{PRODUCT}, parked beside a beautiful beach with palm trees and ocean in the background, {BASE_QUALITY}",
}

environment_images = {}

for name, prompt in environment_prompts.items():
    print("Generating:", name)
    environment_images[name] = generate_ad(
        prompt,
        negative_prompt=DETAILED_NEGATIVE,
        seed=42,
        steps=30,
        guidance=7,
        strength=0.45
    )

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (name, img) in zip(axes.ravel(), environment_images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7B. CAMERA VARIATIONS
# ============================================================
camera_prompts = {
    "Wide-angle": f"{PRODUCT}, photographed with a dramatic wide-angle lens, low perspective, full headphones visible, {BASE_QUALITY}",
    "Close-up": f"{PRODUCT}, close-up three-quarter product view emphasizing ear cups, headband and ear cushions and premium details, {BASE_QUALITY}",
    "Eye-level": f"{PRODUCT}, eye-level professional professional headphone product photography, balanced perspective, {BASE_QUALITY}",
    "Low-angle": f"{PRODUCT}, powerful low-angle camera view from near tabletop level, premium heroic composition, {BASE_QUALITY}",
}

camera_images = {}

for name, prompt in camera_prompts.items():
    print("Generating:", name)
    camera_images[name] = generate_ad(
        prompt,
        negative_prompt=DETAILED_NEGATIVE,
        seed=101,
        steps=30,
        guidance=7,
        strength=0.45
    )

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (name, img) in zip(axes.ravel(), camera_images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7C. LIGHTING VARIATIONS
# ============================================================
lighting_prompts = {
    "Daylight": f"{PRODUCT}, outdoor daylight, bright natural sunlight, clean soft shadows, {BASE_QUALITY}",
    "Golden Hour": f"{PRODUCT}, photographed during golden hour, warm sunlight, long cinematic shadows, {BASE_QUALITY}",
    "Night": f"{PRODUCT}, night-time urban advertisement, dramatic artificial lights, realistic reflections, {BASE_QUALITY}",
    "Studio": f"{PRODUCT}, premium studio lighting, controlled softboxes, clean dark studio background, {BASE_QUALITY}",
}

lighting_images = {}

for name, prompt in lighting_prompts.items():
    print("Generating:", name)
    lighting_images[name] = generate_ad(
        prompt,
        negative_prompt=DETAILED_NEGATIVE,
        seed=202,
        steps=30,
        guidance=7,
        strength=0.45
    )

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (name, img) in zip(axes.ravel(), lighting_images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7D. STYLE VARIATIONS
# ============================================================
style_prompts = {
    "Photorealistic": f"{PRODUCT}, {BASE_QUALITY}, photorealistic DSLR photograph",
    "Cinematic": f"{PRODUCT}, cinematic premium audio-product commercial, anamorphic composition, cinematic depth of field, {BASE_QUALITY}",
    "Luxury": f"{PRODUCT}, luxury premium brand advertisement, elegant composition, sophisticated visual language, {BASE_QUALITY}",
}

style_images = {}

for name, prompt in style_prompts.items():
    print("Generating:", name)
    style_images[name] = generate_ad(
        prompt,
        negative_prompt=DETAILED_NEGATIVE,
        seed=303,
        steps=30,
        guidance=7,
        strength=0.45
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, img) in zip(axes.ravel(), style_images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

# 8. Final Advertisement

This combines environment + camera + lighting + style into one professional advertisement prompt.

In [ ]:
# ============================================================
# 8. FINAL ADVERTISEMENT
# ============================================================
final_prompt = (
    f"{PRODUCT}, parked beside a scenic mountain lake during golden hour, "
    "dramatic low-angle three-quarter front camera view, warm cinematic lighting, "
    "premium luxury audio-product advertising style, subtle depth of field, "
    "realistic reflections on the product surfaces, natural environment, clean composition, "
    "hero product shot, no people, " + BASE_QUALITY
)

final_negative = DETAILED_NEGATIVE

final_image = generate_ad(
    final_prompt,
    negative_prompt=final_negative,
    seed=999,
    steps=40,
    guidance=7,
    strength=0.45
)

show_result("Final AI Product Advertisement", final_image)

# 9. Required Parameter Experiments

The assignment asks to experiment with:
1. Prompt — basic vs detailed
2. Negative Prompt — none vs detailed
3. Seed — 42, 101, 999
4. CFG Scale — 3, 7, 12
5. Inference Steps — 10, 20, 30, 50
6. Image-to-Image Strength — 0.2, 0.45, 0.7
7. Environment — Campus, Mountain, City, Beach
8. Lighting — Daylight, Golden Hour, Night
9. Camera — Wide-angle, Close-up, Eye-level, Low-angle
10. Style — Photorealistic, Cinematic, Luxury

The cells below create representative outputs for each parameter.

In [ ]:
# ============================================================
# 9A. EXPERIMENT 1 — BASIC VS DETAILED PROMPT
# ============================================================
basic_prompt = f"{PRODUCT}, mountain environment"
detailed_prompt = (
    f"{PRODUCT}, parked beside a scenic mountain lake, golden hour, "
    "low-angle three-quarter product view, professional premium audio-product commercial, "
    "realistic reflections, cinematic depth of field, premium luxury advertising, "
    "highly detailed, photorealistic"
)

exp1_basic = generate_ad(basic_prompt, BASIC_NEGATIVE, seed=42, steps=30, guidance=7, strength=0.45)
exp1_detailed = generate_ad(detailed_prompt, DETAILED_NEGATIVE, seed=42, steps=30, guidance=7, strength=0.45)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(exp1_basic); axes[0].set_title("Basic Prompt"); axes[0].axis("off")
axes[1].imshow(exp1_detailed); axes[1].set_title("Detailed Prompt"); axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9B. EXPERIMENT 2 — NEGATIVE PROMPT
# ============================================================
prompt = final_prompt

exp2_none = generate_ad(prompt, negative_prompt="", seed=42, steps=30, guidance=7, strength=0.45)
exp2_detailed = generate_ad(prompt, negative_prompt=DETAILED_NEGATIVE, seed=42, steps=30, guidance=7, strength=0.45)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(exp2_none); axes[0].set_title("No Negative Prompt"); axes[0].axis("off")
axes[1].imshow(exp2_detailed); axes[1].set_title("Detailed Negative Prompt"); axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9C. EXPERIMENT 3 — SEED
# ============================================================
seed_results = {}
for seed in [42, 101, 999]:
    seed_results[seed] = generate_ad(
        final_prompt, DETAILED_NEGATIVE,
        seed=seed, steps=30, guidance=7, strength=0.45
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (seed, img) in zip(axes, seed_results.items()):
    ax.imshow(img); ax.set_title(f"Seed = {seed}"); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9D. EXPERIMENT 4 — CFG SCALE
# ============================================================
cfg_results = {}
for cfg in [3, 7, 12]:
    cfg_results[cfg] = generate_ad(
        final_prompt, DETAILED_NEGATIVE,
        seed=42, steps=30, guidance=cfg, strength=0.45
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (cfg, img) in zip(axes.ravel(), cfg_results.items()):
    ax.imshow(img); ax.set_title(f"CFG Scale = {cfg}"); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9E. EXPERIMENT 5 — INFERENCE STEPS
# ============================================================
steps_results = {}
for steps in [10, 20, 30, 50]:
    steps_results[steps] = generate_ad(
        final_prompt, DETAILED_NEGATIVE,
        seed=42, steps=steps, guidance=7, strength=0.45
    )

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (steps, img) in zip(axes.ravel(), steps_results.items()):
    ax.imshow(img); ax.set_title(f"Inference Steps = {steps}"); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9F. EXPERIMENT 6 — IMAGE-TO-IMAGE STRENGTH
# ============================================================
strength_results = {}
for strength in [0.2, 0.45, 0.7]:
    strength_results[strength] = generate_ad(
        final_prompt, DETAILED_NEGATIVE,
        seed=42, steps=30, guidance=7, strength=strength
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (strength, img) in zip(axes, strength_results.items()):
    ax.imshow(img); ax.set_title(f"Strength = {strength}"); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9G. EXPERIMENT 7 — ENVIRONMENT
# ============================================================
exp7_prompts = {
    "Campus": f"{PRODUCT}, modern university campus, {BASE_QUALITY}",
    "Mountain": f"{PRODUCT}, scenic mountain landscape, {BASE_QUALITY}",
    "City": f"{PRODUCT}, modern city street at night, {BASE_QUALITY}",
    "Beach": f"{PRODUCT}, beautiful tropical beach, {BASE_QUALITY}",
}

exp7 = {}
for name, p in exp7_prompts.items():
    exp7[name] = generate_ad(p, DETAILED_NEGATIVE, seed=42, steps=30, guidance=7, strength=0.45)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (name, img) in zip(axes.ravel(), exp7.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9H. EXPERIMENT 8 — LIGHTING
# ============================================================
exp8_prompts = {
    "Daylight": f"{PRODUCT}, daylight, bright natural sunlight, {BASE_QUALITY}",
    "Golden Hour": f"{PRODUCT}, golden hour, warm sunlight, {BASE_QUALITY}",
    "Night": f"{PRODUCT}, night scene, dramatic artificial lighting, {BASE_QUALITY}",
}

exp8 = {}
for name, p in exp8_prompts.items():
    exp8[name] = generate_ad(p, DETAILED_NEGATIVE, seed=42, steps=30, guidance=7, strength=0.45)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, img) in zip(axes, exp8.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9I. EXPERIMENT 9 — CAMERA
# ============================================================
exp9_prompts = {
    "Wide-angle": f"{PRODUCT}, wide-angle lens, full headphones visible, {BASE_QUALITY}",
    "Close-up": f"{PRODUCT}, close-up three-quarter view, product details emphasized, {BASE_QUALITY}",
    "Eye-level": f"{PRODUCT}, eye-level camera, balanced professional composition, {BASE_QUALITY}",
    "Low-angle": f"{PRODUCT}, dramatic low-angle camera, heroic product composition, {BASE_QUALITY}",
}

exp9 = {}
for name, p in exp9_prompts.items():
    exp9[name] = generate_ad(p, DETAILED_NEGATIVE, seed=42, steps=30, guidance=7, strength=0.45)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (name, img) in zip(axes.ravel(), exp9.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9J. EXPERIMENT 10 — STYLE
# ============================================================
exp10_prompts = {
    "Photorealistic": f"{PRODUCT}, photorealistic DSLR product photograph, {BASE_QUALITY}",
    "Cinematic": f"{PRODUCT}, cinematic premium headphone advertisement, anamorphic look, {BASE_QUALITY}",
    "Luxury": f"{PRODUCT}, luxury premium brand advertisement, elegant sophisticated styling, {BASE_QUALITY}",
}

exp10 = {}
for name, p in exp10_prompts.items():
    exp10[name] = generate_ad(p, DETAILED_NEGATIVE, seed=42, steps=30, guidance=7, strength=0.45)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, img) in zip(axes, exp10.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.tight_layout()
plt.show()

# 10. Comparison Table

The comparison table documents the parameter values used in the experiments. After running the notebook, the images above provide the visual comparison.

In [ ]:
# ============================================================
# 10. COMPARISON TABLE
# ============================================================
comparison = pd.DataFrame([
    [1, "Prompt", "Basic vs Detailed", "Detailed prompt generally gives more controlled composition and scene description."],
    [2, "Negative Prompt", "None vs Detailed", "Detailed negative prompting helps reduce unwanted artifacts and malformed details."],
    [3, "Seed", "42, 101, 999", "Different seeds produce different valid image variations from the same settings."],
    [4, "CFG Scale", "3, 7, 12", "Higher CFG generally follows the text prompt more strongly, but excessive values can look unnatural."],
    [5, "Inference Steps", "10, 20, 30, 50", "More steps usually improve refinement but increase generation time."],
    [6, "Image-to-Image Strength", "0.2, 0.45, 0.7", "Lower strength preserves the reference more; higher strength changes the scene more."],
    [7, "Environment", "Campus, Mountain, City, Beach", "Changes the advertisement context while keeping the product concept."],
    [8, "Lighting", "Daylight, Golden Hour, Night", "Changes mood, shadows, reflections and visual emphasis."],
    [9, "Camera", "Wide-angle, Close-up, Eye-level, Low-angle", "Changes product presentation and visual perspective."],
    [10, "Style", "Photorealistic, Cinematic, Luxury", "Changes the advertising visual language."]
], columns=["Experiment", "Parameter", "Values", "Observation"])

display(comparison)

# 11. Select the Best Combination

For a professional product advertisement, a strong starting combination is:

- Detailed product + scene prompt
- Detailed negative prompt
- Seed = 42 or another visually successful seed
- CFG = 7
- Inference steps = 30–50
- Image-to-image strength = 0.45
- Mountain or city environment
- Golden-hour or controlled studio lighting
- Low-angle or three-quarter camera
- Cinematic/luxury style

The exact best result should be selected from the generated images because Stable Diffusion is stochastic and visual quality can vary with the seed.

In [ ]:
# ============================================================
# 11. BEST PARAMETER COMBINATION
# ============================================================
BEST_PARAMS = {
    "Prompt": "Detailed",
    "Negative Prompt": "Detailed",
    "Seed": 42,
    "CFG Scale": 7,
    "Inference Steps": 40,
    "Image-to-Image Strength": 0.45,
    "Environment": "Mountain",
    "Lighting": "Golden Hour",
    "Camera": "Low-angle",
    "Style": "Cinematic/Luxury"
}

pd.DataFrame(list(BEST_PARAMS.items()), columns=["Parameter", "Selected Value"])

In [ ]:
# ============================================================
# 12. GENERATE THE FINAL BEST ADVERTISEMENT
# ============================================================
best_prompt = (
    f"{PRODUCT}, beside a scenic mountain lake during golden hour, "
    "dramatic low-angle three-quarter product view, cinematic premium headphone advertisement, "
    "luxury premium brand visual language, realistic reflections, natural shadows, "
    "professional DSLR photography, shallow depth of field, highly detailed, "
    "photorealistic, clean premium composition, no people"
)

best_image = generate_ad(
    best_prompt,
    negative_prompt=DETAILED_NEGATIVE,
    seed=42,
    steps=40,
    guidance=7,
    strength=0.45
)

display(best_image)

# Save final deliverable
best_image.save("best_advertisement.png")
print("Saved: best_advertisement.png")

# 13. Save Generated Outputs

The following cell creates an output folder and saves the major experiment results. This makes it easier to submit the generated images along with the Colab notebook.

In [ ]:
# ============================================================
# 13. SAVE MAIN GENERATED RESULTS
# ============================================================
from pathlib import Path

output_dir = Path("assignment3_outputs")
output_dir.mkdir(exist_ok=True)

reference_image.save(output_dir / "01_original_reference.png")
final_image.save(output_dir / "02_final_advertisement.png")
best_image.save(output_dir / "03_best_advertisement.png")

for name, img in environment_images.items():
    img.save(output_dir / f"environment_{name.lower().replace(' ', '_')}.png")

for name, img in camera_images.items():
    img.save(output_dir / f"camera_{name.lower().replace(' ', '_')}.png")

for name, img in lighting_images.items():
    img.save(output_dir / f"lighting_{name.lower().replace(' ', '_')}.png")

for name, img in style_images.items():
    img.save(output_dir / f"style_{name.lower().replace(' ', '_')}.png")

comparison.to_csv(output_dir / "comparison_table.csv", index=False)

print("Saved outputs:")
for p in sorted(output_dir.iterdir()):
    print(" -", p)

In [ ]:
# ============================================================
# 14. OPTIONAL: ZIP OUTPUTS FOR SUBMISSION
# ============================================================
!zip -qr assignment3_outputs.zip assignment3_outputs
print("Created assignment3_outputs.zip")

# 15. Conclusion

Stable Diffusion image-to-image successfully generates multiple professional advertisements from one reference product image. By changing the prompt, negative prompt, seed, CFG scale, inference steps, image-to-image strength, environment, lighting, camera and style, the same headphone product can be presented in substantially different advertising contexts.

The most balanced configuration for this experiment is a detailed prompt with a detailed negative prompt, CFG around 7, 30–50 inference steps and image-to-image strength around 0.45. Lower strength preserves the original product more strongly, while higher strength produces larger visual changes. The final choice should be based on the generated results and the degree of product identity preservation required.